In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:
data=pd.read_csv("Bike/day.csv")
data.head()
#predecir demanda demanda de bicicletas Mediante Regresion Lineal
data.info()

Columna	¿Por qué usar One-Hot?
season	Los números 1, 2, 3, 4 son etiquetas (Primavera, Verano...). Un modelo podría pensar erróneamente que el "4" es más importante o "mayor" que el "1", cuando solo son estaciones diferentes.
weathersit	Representa el clima (1: Despejado, 2: Niebla, 3: Lluvia). El One-Hot ayuda a separar el efecto de la lluvia del efecto de un día soleado.
weekday	Transforma el día de la semana (0-6) en columnas independientes para que el modelo aprenda, por ejemplo, que los domingos se alquilan menos bicis que los lunes.
mnth	Los meses (1-12) también se benefician de esto para capturar la estacionalidad de forma más precisa

In [ ]:
"""
def one_hot(df, column):
    #Codificación one-hot manual para variables categóricas.
    uniques = sorted(df[column].unique())
    for val in uniques[1:]:  # drop_first=True
        df[f"{column}_{val}"] = (df[column] == val).astype(float)
    df.drop(column, axis=1, inplace=True)
    return df
"""

Usamos Encoder para usar el oneHot de sklearn

In [ ]:
encoder = OneHotEncoder(drop='first', sparse_output=False)
columnasCategoricas = ['season', 'yr', 'mnth', 'holiday', 'weekday']

datasetCodificado = encoder.fit_transform(data[columnasCategoricas])

print(datasetCodificado)  # Mostrar el array
print(datasetCodificado.shape)

Normalizamos datos

In [ ]:
def normalizarCaracteristicas(x):
    x_norm = x.copy()
    mu = np.zeros(x.shape[1])
    sigma = np.zeros(x.shape[1])
    
    mu = np.mean(x, axis=0)
    sigma = np.std(x, axis=0)
    x_norm = (x - mu) / sigma
    return x_norm,mu,sigma

In [ ]:
# Extraer las columnas numéricas
caracteristicasNumericas = data[['temp', 'atemp', 'hum', 'windspeed']].values

# Normalizarlas
caracteristicasNormalizadas, mu, sigma = normalizarCaracteristicas(caracteristicasNumericas)

# Normalizar Y también
y_datos = data['cnt'].values.reshape(-1, 1)
y_normalizada, mu_y, sigma_y = normalizarCaracteristicas(y_datos)
y_normalizada = y_normalizada.flatten()  # Volver a 1D

# IMPORTANTE: Convertir a escalares
mu_y = mu_y[0]
sigma_y = sigma_y[0]

# Combinar X
X = np.hstack([datasetCodificado, caracteristicasNormalizadas])

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_normalizada, test_size=0.2, random_state=42)